In [1]:
import ebm_digits

In [3]:
from pathlib import Path
print(Path.cwd())
print((Path.cwd() / "runs" / "ebm_checkpoint.pt").resolve())
print((Path.cwd() / "runs" / "ebm_checkpoint.pt").exists())

c:\Users\vince\Documents\master\Studium\sem3\RAML\code\ebm-digits\notebooks
C:\Users\vince\Documents\master\Studium\sem3\RAML\code\ebm-digits\notebooks\runs\ebm_checkpoint.pt
False


In [ ]:
import torch
from ebm_digits.model import EBM, EnergyCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
root = Path(r"c:\Users\vince\Documents\master\Studium\sem3\RAML\code\ebm-digits")
ckpt_path = root / "runs" / "ebm_checkpoint.pt"

assert ckpt_path.is_file(), f"Missing {ckpt_path} — run training first: uv run ebm-train"

ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
cfg = ckpt["ebm_config"]

energy_net = EnergyCNN().to(device)
energy_net.load_state_dict(ckpt["energy_net"])

ebm = EBM(
    energy_net,
    alpha=float(cfg["alpha"]),          # can match cfg["alpha"] or tune for sampling
    sigma=float(cfg["sigma"]),
    ld_steps=2000,        # often more steps at sample time than during training
    image_shape=tuple(cfg["image_shape"]),
).to(device)
ebm.eval()

with torch.no_grad():
    digits = ebm.sample(batch_size=64)   # shape [64, 1, 28, 28], values in [-1, 1]

In [6]:
from torchvision.utils import save_image

imgs = (digits.cpu() + 1) * 0.5
save_image(imgs, "samples.png", nrow=8)